<div style="text-align: center;">
  <span style="
      color: lightskyblue;
      font-family: 'Segoe UI', 'Noto Sans KR', 'Malgun Gothic', sans-serif;
      font-weight: 700;
      font-size: 48px;
      letter-spacing: 0.5px;
      line-height: 1.2;
  ">
    Demo File 
  </span>
</div>

<div style="color:white; font-size:16px; font-weight:300; line-height:1.5;">
This is the Demo File for 260826<br><br>
Scenario:<br>
&nbsp;&nbsp;&nbsp;&nbsp;1. In 1st Floor Laboratory, the robot turns left 90 degrees and takes the leftmost black block<br><br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(1) If the leftmost black block is in left side from the center of robot:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Use its left arm.<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(2) If the leftmost black block is in right side from the center of robot:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Use its right arm.<br><br>
&nbsp;&nbsp;&nbsp;&nbsp;2. The robot turns right 90 degrees and check the yellow collecting box's position<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(1) If the box is in the left side, and the block is in the left arm:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Lay the block by using current(left) arm.<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(2) If the box is in the right side, and the block is in the left arm:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Transfers the object from the left arm to the right arm. Lay the block by using tranferred(right) arm.<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(3) If the box is in the left side, and the block is in the right arm:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Transfers the object from the right arm to the left arm. Lay the block by using tranferred(left) arm.<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(4) If the box is in the right side, and the block is in the right arm:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Lay the block by using current(right) arm.<br><br>
If you want to use Only one specific case (left arm grasp → left-to-right transfer -> lay to right box),<br>
use <code>demoFile_pickAndPlace.ipynb</code>.
</div>

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import logging
import time
import pyrealsense2 as rs
import cv2

import sys
from pathlib import Path
# Path.cwd()           : folder "260723", 
# Path.cwd().parents[0]: folder "Demonstration", 
# Path.cwd().parents[1]: folder "RBY1_library_ysw"
sys.path.append(str(Path.cwd().parents[1]))                  
import rby1_lib_ysw as ysw

np.set_printoptions(suppress=True, precision=4, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Initialization [Connect + PowerON]
</span>

In [2]:
# ==========================================================================================================
# Robot Initialization
# ==========================================================================================================
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot + UPC

robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
if (robot != None):
    model = robot.model()
    
time.sleep(1)


# ==========================================================================================================
# Gripper Initialization
# Gripper Setting (Need to hard check of gripper ping test)
# ==========================================================================================================
gripper = ysw.Gripper()
gripper.connect_hardCheck(
    device_name=ysw.GRIPPER_DEVICE_NAME, 
    protocol_version=ysw.DXL_PROTOCOL_VERSION, 
    baudrate=ysw.DXL_BAUDRATE,
    retry_interval=1.0,
    max_retry=5
)


# ==========================================================================================================
# Depth Camera Initialization
# D435i:  Head
# D405_R: Right Wrist
# D405_L: Left Wrist
# ==========================================================================================================
d435i = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.HEAD_D435I_SERIAL)
d435i.initialize()
d405_R = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.RIGHT_D405_SERIAL)
d405_R.initialize()
d405_L = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(640,480), fps=30, serial_number=ysw.LEFT_D405_SERIAL)
d405_L.initialize()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!
[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************



*********************** [Gripper Hard Connection Starts] ***********************

[Gripper] Trying to connect gripper... attempt=1
************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Do

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Information
</span>

In [3]:
# case_list
# (grasp side, box side)
# (grasp_arm, lay_arm)
# "left-to-left": grasp using left arm and lay using left arm
# "left_to_right"
# "right_to_left" 
# "right_to_right"


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Tuning Parameters
</span>

In [3]:
position_control_minimum_time = 5.5      # [sec]

# Move to Table
vision_record_duration_tableDistance = 5.0    # [sec]
offset_table_distance = 0.37                  # [m]
vision_record_duration_checkBox = 5.0         # [sec]
vision_record_duration_checkObject = 5.0      # [sec]
box_minimum_area = 8000                       # [pixel^2]
box_maximum_distance = 1.2                    # [m]
object_minimum_area = 50                      # [pixel^2]
object_maximum_area = 6000                    # [pixel^2]
object_maximum_distance = 1.2                 # [m]

# minimum_time_scaler = 0.5                # For short movement
# grasp_current = 0.2     # [A]
# object_floating_x_in_linktorso5 = -0.05   # [m]
# object_floating_z_in_linktorso5 = 0.05    # [m]
# vision_record_duration = 3.0    # [sec]

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [Initial]
</span>

In [5]:
desired_pos = ysw.PACK_POS  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:gray;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Table Distance Detection [Visualization]
</span>

In [ ]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: PACK_POS
# ============================================================
width = 640
height = 480
pixel_roi_x = [140, width - 140]      
pixel_roi_y = [320, height]    
# pixel_roi_x = None            # None이면 x 전체 사용
# pixel_roi_y = None            # None이면 y 전체 사용
    
    
# ============================================================
# Get the Distance of Center Pixel 
# Table distance is approximated as the depth of center pixel
# ============================================================

d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)
        
        h, w, _ = color_image.shape
        
    
        # ROI center pixel in full image coordinate
        if pixel_roi_x is None:
            x_min = 0
            x_max = w
        else:
            x_min, x_max = pixel_roi_x
            x_min = int(np.clip(x_min, 0, w - 1))
            x_max = int(np.clip(x_max, 0, w))

        if pixel_roi_y is None:
            y_min = 0
            y_max = h
        else:
            y_min, y_max = pixel_roi_y
            y_min = int(np.clip(y_min, 0, h - 1))
            y_max = int(np.clip(y_max, 0, h))
            
        x_roi_center = (x_min + x_max) // 2
        y_roi_center = (y_min + y_max) // 2

        distance = d435i.getDepth(depth_frame, (x_roi_center, y_roi_center))   # unit = [m]
        
        
        # Draw ROI box on full image
        cv2.rectangle(
            color_image,
            (x_min, y_min),
            (x_max, y_max),
            (255, 0, 0),
            2
        )
        cv2.putText(
            color_image,                   
            f"ROI Box",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )
        
        # Draw ROI center point on full image
        cv2.circle(color_image, (x_roi_center, y_roi_center), 5, (0, 0, 255), -1)
        cv2.putText(
            color_image,                   
            f"[ROI center point] Depth: {distance:.3f} m",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Table Distance Detection [5 seconds]
</span>

In [7]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: PACK_POS
# ============================================================
width = 640
height = 480
pixel_roi_x = [140, width - 140]      
pixel_roi_y = [320, height]    
# pixel_roi_x = None            # None이면 x 전체 사용
# pixel_roi_y = None            # None이면 y 전체 사용
    
    
# ============================================================
# Get the Distance of Center Pixel 
# Table distance is approximated as the depth of center pixel
# ============================================================

record_duration = vision_record_duration_tableDistance      # [sec]
distance_list = []

d435i.startPipeline()

try:
    start_time = time.time()
    
    while True:
        elapsed_time = time.time() - start_time
        
        if elapsed_time >= record_duration:
            break
        
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        h, w, _ = color_image.shape
        
        # ROI center pixel in full image coordinate
        if pixel_roi_x is None:
            x_min = 0
            x_max = w
        else:
            x_min, x_max = pixel_roi_x
            x_min = int(np.clip(x_min, 0, w - 1))
            x_max = int(np.clip(x_max, 0, w))

        if pixel_roi_y is None:
            y_min = 0
            y_max = h
        else:
            y_min, y_max = pixel_roi_y
            y_min = int(np.clip(y_min, 0, h - 1))
            y_max = int(np.clip(y_max, 0, h))
            
        x_roi_center = (x_min + x_max) // 2
        y_roi_center = (y_min + y_max) // 2

        distance = d435i.getDepth(depth_frame, (x_roi_center, y_roi_center))   # unit = [m]
        distance_list.append(distance)
        print(f"Table Distance: {distance:.3f} [m]",)

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()
    

# Average Distance
distance_avg = np.mean(distance_list)
print(f"\nTable Average Distance: {distance_avg} [m]")

moved_distance_to_table = distance_avg - offset_table_distance
print(f"moved_distance_to_table: {moved_distance_to_table} [m]")

[INFO] [Head D435i] Pipeline started.
Table Distance: 1.612 [m]
Table Distance: 1.638 [m]
Table Distance: 1.600 [m]
Table Distance: 1.634 [m]
Table Distance: 1.604 [m]
Table Distance: 1.634 [m]
Table Distance: 1.656 [m]
Table Distance: 1.651 [m]
Table Distance: 1.643 [m]
Table Distance: 1.617 [m]
Table Distance: 1.638 [m]
Table Distance: 1.621 [m]
Table Distance: 1.596 [m]
Table Distance: 1.612 [m]
Table Distance: 1.643 [m]
Table Distance: 1.608 [m]
Table Distance: 1.643 [m]
Table Distance: 1.617 [m]
Table Distance: 1.638 [m]
Table Distance: 1.643 [m]
Table Distance: 1.604 [m]
Table Distance: 1.647 [m]
Table Distance: 1.630 [m]
Table Distance: 1.638 [m]
Table Distance: 1.617 [m]
Table Distance: 1.617 [m]
Table Distance: 1.630 [m]
Table Distance: 1.634 [m]
Table Distance: 1.634 [m]
Table Distance: 1.621 [m]
Table Distance: 1.604 [m]
Table Distance: 1.630 [m]
Table Distance: 1.634 [m]
Table Distance: 1.647 [m]
Table Distance: 1.612 [m]
Table Distance: 1.612 [m]
Table Distance: 1.643 [m]


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [DEMO POSE]
</span>

In [4]:
desired_pos = ysw.DEMO_BASIC_POS_260826  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to Table
</span>

In [9]:
ysw.mobile_linear_distance(
    robot,
    distance_m=moved_distance_to_table,
    max_speed=0.25,
    accel=0.8,
    tolerance_m=0.01
)

distance=0.000[m], speed=0.016[m/s], pos_error=1.261[m], yaw_error=0.00[deg]
distance=0.010[m], speed=0.144[m/s], pos_error=1.251[m], yaw_error=-0.02[deg]
distance=0.051[m], speed=0.250[m/s], pos_error=1.209[m], yaw_error=-0.01[deg]
distance=0.103[m], speed=0.250[m/s], pos_error=1.157[m], yaw_error=-0.11[deg]
distance=0.156[m], speed=0.250[m/s], pos_error=1.105[m], yaw_error=0.08[deg]
distance=0.208[m], speed=0.250[m/s], pos_error=1.053[m], yaw_error=0.09[deg]
distance=0.260[m], speed=0.250[m/s], pos_error=1.001[m], yaw_error=-0.00[deg]
distance=0.312[m], speed=0.250[m/s], pos_error=0.949[m], yaw_error=-0.01[deg]
distance=0.365[m], speed=0.250[m/s], pos_error=0.896[m], yaw_error=-0.03[deg]
distance=0.417[m], speed=0.250[m/s], pos_error=0.844[m], yaw_error=-0.01[deg]
distance=0.469[m], speed=0.250[m/s], pos_error=0.792[m], yaw_error=-0.01[deg]
distance=0.521[m], speed=0.250[m/s], pos_error=0.739[m], yaw_error=-0.02[deg]
distance=0.573[m], speed=0.250[m/s], pos_error=0.688[m], yaw_error=

True

<span style="
    color:gray;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check Black & Red Boxes and Objects [Visualization]
</span>

In [12]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: DEMO_BASIC_POSE_260826
# ============================================================
width = 640
height = 480
pixel_roi_x = None            # None이면 x 전체 사용
pixel_roi_y = [150, height]  


# ============================================================
# Get the Information about Black & Red Collecting Boxes
# ============================================================

rv = d435i.startPipeline()

# if rv is False:
#     print(f"[WARNING] Camera start failed. Try recovery...")
#     rv = d435i.recoverPipeline(
#         align_enable=True,
#         try_hardware_reset=True,
#         wait_after_reset_sec=5.0,
#     )
# if rv is False:
#     raise RuntimeError(f"Camera Recovery failed.")   

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        # ----------------------------------------------------
        # Keep raw image for detection
        # Do not detect on already-annotated image
        # ----------------------------------------------------
        color_image_raw = color_image.copy()
        
        detected_blackBoxes, color_image_forBlackBox, mask_blackBox = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK_BOX,
            minimum_area=box_minimum_area,       
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )
        detected_redBoxes, color_image_forRedBox, mask_redBox = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_RED_BOX,
            minimum_area=box_minimum_area,       
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )
        detected_blackObjects, color_image_forBlackObj, mask_blackObj = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK_BOX,
            minimum_area=object_minimum_area, 
            maximum_area=object_maximum_area,      
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )
        detected_redObjects, color_image_forRedObj, mask_redObj = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_RED,
            minimum_area=object_minimum_area, 
            maximum_area=object_maximum_area,         
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )
        
        for obj in detected_blackBoxes:
            print(
                "[Black Box]: "
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
                f"area_2d={obj['area_2d']:.1f}"
            )
        for obj in detected_redBoxes:
            print(
                "[Red Box]: "
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}, "
                f"area_2d={obj['area_2d']:.1f}"
            )
        for obj in detected_blackObjects:
            print(
                "[Black Objects]: "
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
                f"area_2d={obj['area_2d']:.1f}"
            )
        for obj in detected_redObjects:
            print(
                "[Orange Objects]: "
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
                f"area_2d={obj['area_2d']:.1f}"
            )
        
        d435i.imshow("blackBox_detection", color_image_forBlackBox)
        d435i.imshow("redBox_detection", color_image_forRedBox)
        d435i.imshow("blackObj_detection", color_image_forBlackObj)
        d435i.imshow("orangeObj_detection", color_image_forRedObj)


        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] [Head D435i] Pipeline started.


[Black Box]: [object 0] center_3d=[-0.18   0.105  0.664], bbox_3d_size=[0.179 0.192 0.157]area_2d=24601.5
[Black Objects]: [object 0] center_3d=[0.266 0.141 0.643], bbox_3d_size=[0.105 0.035 0.042]area_2d=2268.5
[Black Objects]: [object 1] center_3d=[0.152 0.121 0.684], bbox_3d_size=[0.015 0.011 0.007]area_2d=111.0
[Black Objects]: [object 2] center_3d=[0.002 0.118 0.655], bbox_3d_size=[0.038 0.043 0.09 ]area_2d=1198.5
[Black Objects]: [object 3] center_3d=[0.155 0.061 0.74 ], bbox_3d_size=[0.013 0.012 0.006]area_2d=85.5
[Black Objects]: [object 4] center_3d=[-0.069  0.046  0.728], bbox_3d_size=[0.043 0.05  0.02 ]area_2d=1121.0
[Black Objects]: [object 5] center_3d=[0.014 0.044 0.739], bbox_3d_size=[0.041 0.049 0.024]area_2d=1055.0
[Black Box]: [object 0] center_3d=[-0.192  0.101  0.67 ], bbox_3d_size=[0.129 0.19  0.245]area_2d=22051.0
[Red Box]: [object 0] center_3d=[0.29  0.059 0.71 ], bbox_3d_size=[0.125 0.145 0.149], area_2d=10621.5
[Black Objects]: [object 0] center_3d=[0.261 0.14

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check [Black & Red] Boxes [5 seconds]
</span>

In [ ]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: DEMO_BASIC_POSE_260826
# ============================================================
width = 640
height = 480
pixel_roi_x = None            # None이면 x 전체 사용
pixel_roi_y = [150, height]  



# ============================================================
# Get the Information about Black & Red Boxes and Objects
# ============================================================
record_duration = vision_record_duration_checkObject  # [sec]

blackBox_samples = []
redBox_samples = []

d435i.startPipeline()

try:
    start_time = time.time()

    while True:
        elapsed_time = time.time() - start_time

        if elapsed_time >= record_duration:
            break

        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)

        # Detection용 원본 이미지 보존
        color_image_raw = color_image.copy()

        detected_blackBoxes, color_image_forBlackBox, mask_blackBox = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK_BOX,
            minimum_area=box_minimum_area,       
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )
        detected_redBoxes, color_image_forRedBox, mask_redBox = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_RED_BOX,
            minimum_area=box_minimum_area,       
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )

        if len(detected_blackBoxes) > 0:
            largest_blackBox = max(
                detected_blackBoxes,
                key=lambda obj: obj["area_2d"]
            )
            # [Largest area] sample
            blackBox_samples.append({
                "area_2d":      float(largest_blackBox["area_2d"]),
                "pixel_center": np.array(largest_blackBox["pixel_center"], dtype=np.float64),
                "center_3d":    np.array(largest_blackBox["center_3d"], dtype=np.float64),
                "bbox_3d_size": np.array(largest_blackBox["bbox_3d_size"], dtype=np.float64),
                "bbox_3d_min":  np.array(largest_blackBox["bbox_3d_min"], dtype=np.float64),
                "bbox_3d_max":  np.array(largest_blackBox["bbox_3d_max"], dtype=np.float64),
            })
            
        if len(detected_redBoxes) > 0:
            largest_redBox = max(
                detected_redBoxes,
                key=lambda obj: obj["area_2d"]
            )
            # [Largest area] sample
            redBox_samples.append({
                "area_2d":      float(largest_redBox["area_2d"]),
                "pixel_center": np.array(largest_redBox["pixel_center"], dtype=np.float64),
                "center_3d":    np.array(largest_redBox["center_3d"], dtype=np.float64),
                "bbox_3d_size": np.array(largest_redBox["bbox_3d_size"], dtype=np.float64),
                "bbox_3d_min":  np.array(largest_redBox["bbox_3d_min"], dtype=np.float64),
                "bbox_3d_max":  np.array(largest_redBox["bbox_3d_max"], dtype=np.float64),
            })

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()



# =========================
# Average Box
# =========================

blackBox_sample_avg = None
redBox_sample_avg = None

if len(blackBox_samples) > 0:
    blackBox_sample_avg = ysw.average_detectColorObjectContour3D(blackBox_samples)
if len(redBox_samples) > 0:
    redBox_sample_avg = ysw.average_detectColorObjectContour3D(redBox_samples)

print("\n========== Black & Red box Detection Result ==========")

if blackBox_sample_avg is None:
    print("No Black Box detected.")
else:
    print("\n[Black Box]")
    print(f"num_samples             : {blackBox_sample_avg['num_samples']}")
    print(f"area_2d_avg             : {blackBox_sample_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {blackBox_sample_avg['area_2d_median']:.1f}")
    print(f"center_3d_avg           : {np.round(blackBox_sample_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(blackBox_sample_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(blackBox_sample_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(blackBox_sample_avg['bbox_3d_size_median'], 4)} m")
    
if redBox_sample_avg is None:
    print("No Red Box detected.")
else:
    print("\n[Red Box]")
    print(f"num_samples             : {redBox_sample_avg['num_samples']}")
    print(f"area_2d_avg             : {redBox_sample_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {redBox_sample_avg['area_2d_median']:.1f}")
    print(f"center_3d_avg           : {np.round(redBox_sample_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(redBox_sample_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(redBox_sample_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(redBox_sample_avg['bbox_3d_size_median'], 4)} m")   


[INFO] [Head D435i] Pipeline started.
[INFO] [Head D435i] Pipeline stopped.

========== Detection Result ==========
Only the Leftmost object is used


[Leftmost Black Object]
num_samples             : 101
area_2d_avg             : 982.6
area_2d_median          : 974.0
center_3d_avg           : [-0.0819  0.1044  0.6631] m
center_3d_median        : [-0.0821  0.105   0.662 ] m
bbox_3d_size_avg        : [0.0392 0.0376 0.0329] m
bbox_3d_size_median     : [0.0392 0.0374 0.033 ] m

[Leftmost Black Object]
num_samples             : 101
area_2d_avg             : 755.3
area_2d_median          : 787.5
center_3d_avg           : [0.0657 0.1039 0.6758] m
center_3d_median        : [0.0656 0.1069 0.672 ] m
bbox_3d_size_avg        : [0.0324 0.0342 0.0329] m
bbox_3d_size_median     : [0.0325 0.0348 0.034 ] m


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check [Black & Red] Leftmost Object [5 seconds]
</span>

In [ ]:
# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# Posture: DEMO_BASIC_POSE_260826
# ============================================================
width = 640
height = 480
pixel_roi_x = None            # None이면 x 전체 사용
pixel_roi_y = [150, height]  



# ============================================================
# Get the Information about Black & Red Boxes and Objects
# ============================================================
record_duration = vision_record_duration_checkBox  # [sec]

leftmost_blackObj_samples = []
leftmost_redObj_samples = []

d435i.startPipeline()

try:
    start_time = time.time()

    while True:
        elapsed_time = time.time() - start_time

        if elapsed_time >= record_duration:
            break

        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)

        # Detection용 원본 이미지 보존
        color_image_raw = color_image.copy()

        detected_blackObjects, color_image_forBlackObj, mask_blackObj = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK_BOX,
            minimum_area=object_minimum_area, 
            maximum_area=object_maximum_area,      
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )
        detected_redObjects, color_image_forRedObj, mask_redObj = d435i.detectColorObjectContour3D(
            color_image=color_image_raw.copy(),
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_RED,
            minimum_area=object_minimum_area, 
            maximum_area=object_maximum_area,         
            maximum_depth=box_maximum_distance,                   # ignore distant object larger than x[m] 
            pixel_roi_x=pixel_roi_x,
            pixel_roi_y=pixel_roi_y,
        )

        if len(detected_blackObjects) > 0:
            # 카메라 기준 X 좌표가 작을수록 이미지 왼쪽
            # RealSense/D435i 기준: X+ = image right
            blackObj_list_fromLeft = sorted(
                detected_blackObjects,
                key=lambda obj: obj["center_3d"][0],
                reverse=False
            )

            # 제일 왼쪽 물체 하나만 선택
            obj = blackObj_list_fromLeft[0]

            # 제일 왼쪽 물체 sample만 저장
            leftmost_blackObj_samples.append({
                "area_2d":      float(obj["area_2d"]),
                "pixel_center": np.array(obj["pixel_center"], dtype=np.float64),
                "center_3d":    np.array(obj["center_3d"], dtype=np.float64),
                "bbox_3d_size": np.array(obj["bbox_3d_size"], dtype=np.float64),
                "bbox_3d_min":  np.array(obj["bbox_3d_min"], dtype=np.float64),
                "bbox_3d_max":  np.array(obj["bbox_3d_max"], dtype=np.float64),
            })
        
        if len(detected_redObjects) > 0:
            # 카메라 기준 X 좌표가 작을수록 이미지 왼쪽
            # RealSense/D435i 기준: X+ = image right
            redObj_list_fromLeft = sorted(
                detected_redObjects,
                key=lambda obj: obj["center_3d"][0],
                reverse=False
            )

            # 제일 왼쪽 물체 하나만 선택
            obj = redObj_list_fromLeft[0]

            # 제일 왼쪽 물체 sample만 저장
            leftmost_redObj_samples.append({
                "area_2d":      float(obj["area_2d"]),
                "pixel_center": np.array(obj["pixel_center"], dtype=np.float64),
                "center_3d":    np.array(obj["center_3d"], dtype=np.float64),
                "bbox_3d_size": np.array(obj["bbox_3d_size"], dtype=np.float64),
                "bbox_3d_min":  np.array(obj["bbox_3d_min"], dtype=np.float64),
                "bbox_3d_max":  np.array(obj["bbox_3d_max"], dtype=np.float64),
            })

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

    
    
# =========================
# Average leftmost objects
# =========================
leftmost_blackObj_sample_avg = None
leftmost_redObj_sample_avg = None

if len(leftmost_blackObj_samples) > 0:
    leftmost_blackObj_sample_avg = ysw.average_detectColorObjectContour3D(leftmost_blackObj_samples)
if len(leftmost_redObj_samples) > 0:
    leftmost_redObj_sample_avg = ysw.average_detectColorObjectContour3D(leftmost_redObj_samples)

print("\n========== Detection Result ==========")
print("Only the Leftmost object is used\n")

if leftmost_blackObj_sample_avg is None:
    print("No Black Object detected.")
else:
    print("\n[Leftmost Black Object]")
    print(f"num_samples             : {leftmost_blackObj_sample_avg['num_samples']}")
    print(f"area_2d_avg             : {leftmost_blackObj_sample_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {leftmost_blackObj_sample_avg['area_2d_median']:.1f}")
    print(f"center_3d_avg           : {np.round(leftmost_blackObj_sample_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(leftmost_blackObj_sample_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(leftmost_blackObj_sample_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(leftmost_blackObj_sample_avg['bbox_3d_size_median'], 4)} m")
    
if leftmost_redObj_samples is None:
    print("No Red Object detected.")
else:
    print("\n[Leftmost Black Object]")
    print(f"num_samples             : {leftmost_redObj_sample_avg['num_samples']}")
    print(f"area_2d_avg             : {leftmost_redObj_sample_avg['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {leftmost_redObj_sample_avg['area_2d_median']:.1f}")
    print(f"center_3d_avg           : {np.round(leftmost_redObj_sample_avg['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(leftmost_redObj_sample_avg['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(leftmost_redObj_sample_avg['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(leftmost_redObj_sample_avg['bbox_3d_size_median'], 4)} m") 